<a href="https://colab.research.google.com/github/NazmulHudaNabil/banglish-to-bangla/blob/main/Banglish_to_Bangla_transliteration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U unsloth
!pip install -U datasets trl transformers accelerate bitsandbytes

In [ ]:
import os
import torch
import random
import numpy as np

from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

# Reproducibility
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:

# Model and training configuration
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512
OUTPUT_DIR = "banglish-to-bangla-qwen2.5"
MAX_NEW_TOKENS = 256

# Load a 4-bit model for QLoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,              # Automatically select a supported dtype
    load_in_4bit=True,
)

# Padding configuration
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Model loaded successfully.")
print("Context length:", MAX_SEQ_LENGTH)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model

In [ ]:
tokenizer

In [ ]:
import torch
torch.cuda.memory_summary()

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   1134 MiB |   1166 MiB |   2250 MiB |   1116 MiB |
|       from large pool |   1103 MiB |   1135 MiB |   2219 MiB |   1116 MiB |
|       from small pool |     31 MiB |     31 MiB |     31 MiB |      0 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   1134 MiB |   1166 MiB |   2250 MiB |   1116 MiB |
|       from large pool |   1103 MiB |   1135 MiB |   2219 MiB |   1116 MiB |
|       from small pool |     31 MiB |     31 MiB |     31 MiB |      0 MiB |
|---------------------------------------------------------------------------|
| Requested memory      |   1134 MiB |   1166 MiB |   2250 MiB |   1116 MiB |
|       from large pool |   1103 MiB |   1135 MiB |   2219 MiB |   1116 MiB |
|       from small pool |     30 MiB |     30 MiB |     30 MiB |      0 MiB |
|---------------------------------------------------------------------------|
| GPU reserved memory   |   1172 MiB |   1172 MiB |   1172 MiB |      0 B   |
|       from large pool |   1140 MiB |   1140 MiB |   1140 MiB |      0 B   |
|       from small pool |     32 MiB |     32 MiB |     32 MiB |      0 B   |
|---------------------------------------------------------------------------|
| Non-releasable memory |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Allocations           |    1325    |    1327    |    1534    |     209    |
|       from large pool |     146    |     148    |     153    |       7    |
|       from small pool |    1179    |    1180    |    1381    |     202    |
|---------------------------------------------------------------------------|
| Active allocs         |    1325    |    1327    |    1534    |     209    |
|       from large pool |     146    |     148    |     153    |       7    |
|       from small pool |    1179    |    1180    |    1381    |     202    |
|---------------------------------------------------------------------------|
| GPU reserved segments |       0    |       0    |       0    |       0    |
|       from large pool |       0    |       0    |       0    |       0    |
|       from small pool |       0    |       0    |       0    |       0    |
|---------------------------------------------------------------------------|
| Non-releasable allocs |       0    |       0    |       0    |       0    |
|       from large pool |       0    |       0    |       0    |       0    |
|       from small pool |       0    |       0    |       0    |       0    |
|---------------------------------------------------------------------------|
| Oversize allocations  |       0    |       0    |       0    |       0    |
|---------------------------------------------------------------------------|
| Oversize GPU segments |       0    |       0    |       0    |       0    |
|===========================================================================|


In [ ]:
DATASET_FILES = {
    "train": (
        "https://huggingface.co/datasets/aplycaebous/BanglaTLit/"
        "resolve/main/BanglaTLit_train.csv"
    ),
    "validation": (
        "https://huggingface.co/datasets/aplycaebous/BanglaTLit/"
        "resolve/main/BanglaTLiT_val.csv"
    ),
    "test": (
        "https://huggingface.co/datasets/aplycaebous/BanglaTLit/"
        "resolve/main/BanglaTLiT_test.csv"
    ),
}

dataset = load_dataset(
    "csv",
    data_files=DATASET_FILES,
)

print(dataset)
print("Training columns:", dataset["train"].column_names)
print("Example:", dataset["train"][0])

In [ ]:
print("Tokenizer:", tokenizer.name_or_path)
print("Chat template exists:", tokenizer.chat_template is not None)
print("EOS token:", repr(tokenizer.eos_token))
print("Pad token:", repr(tokenizer.pad_token))

print("Dataset columns:", dataset["train"].column_names)
print("First row:", dataset["train"][0])

## Format the data for instruction fine-tuning

In [ ]:
SYSTEM_INSTRUCTION = (
    "Convert Banglish (Romanized Bangla) into natural Bangla "
    "written in Bengali script. Preserve the original meaning, "
    "names, numbers, and tone. Output only the Bangla conversion. "
    "Do not explain your answer."
)

def format_example(example):
    # Convert any missing (None) values to empty strings to avoid TypeError
    user_text = example["text_transliterated"] if example["text_transliterated"] is not None else ""
    assistant_text = example["text_bengali"] if example["text_bengali"] is not None else ""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_INSTRUCTION,
        },
        {
            "role": "user",
            "content": str(user_text),
        },
        {
            "role": "assistant",
            "content": str(assistant_text),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text + tokenizer.eos_token}

# Filter out any rows with completely null values from the splits to keep clean data
dataset = dataset.filter(lambda x: x["text_transliterated"] is not None and x["text_bengali"] is not None)

formatted_dataset = dataset.map(
    format_example,
    desc="Formatting Banglish-Bangla examples",
)

print(formatted_dataset["train"][0]["text"])

In [ ]:
formatted_dataset["train"][0]

{'id': 'f5afeacc-e452-497d-80d4-daf5c3238368',
 'text_transliterated': 'oi mia watermark a click korle subscribe hobe na?? title edit koren vaotabaji kothakar',
 'text_bengali': 'ঐ মিয়া ওয়াটারমার্ক এ ক্লিক করলে সাবস্ক্রাইব হবে না? টাইটেল এডিট করেন ভাওতাবাজি কোথাকার',
 'text': '<|im_start|>system\nConvert Banglish (Romanized Bangla) into natural Bangla written in Bengali script. Preserve the original meaning, names, numbers, and tone. Output only the Bangla conversion. Do not explain your answer.<|im_end|>\n<|im_start|>user\noi mia watermark a click korle subscribe hobe na?? title edit koren vaotabaji kothakar<|im_end|>\n<|im_start|>assistant\nঐ মিয়া ওয়াটারমার্ক এ ক্লিক করলে সাবস্ক্রাইব হবে না? টাইটেল এডিট করেন ভাওতাবাজি কোথাকার<|im_end|>\n<|im_end|>'}

## Filter examples that exceed 512 tokens

In [ ]:
def add_length(example_batch):
    encoded = tokenizer(
        example_batch["text"],
        add_special_tokens=False,
        truncation=False,
    )
    lengths = [len(ids) for ids in encoded["input_ids"]]

    return {
        "token_length": lengths,
        "keep": [length <= MAX_SEQ_LENGTH for length in lengths],
    }


processed_dataset = {}

for split_name, split_data in formatted_dataset.items():
    checked = split_data.map(
        add_length,
        batched=True,
        batch_size=256,
        desc=f"Checking {split_name} lengths",
    )

    original_count = len(checked)
    checked = checked.filter(
        lambda row: row["keep"],
        desc=f"Filtering {split_name}",
    )

    checked = checked.remove_columns(["token_length", "keep"])
    processed_dataset[split_name] = checked

    print(
        f"{split_name}: retained {len(checked):,} "
        f"of {original_count:,} examples"
    )

train_dataset = processed_dataset["train"]
validation_dataset = processed_dataset["validation"]
test_dataset = processed_dataset["test"]

assert len(train_dataset) > 0
assert len(validation_dataset) > 0

In [ ]:
train_dataset = processed_dataset["train"]
validation_dataset = processed_dataset["validation"]
test_dataset = processed_dataset["test"]


print(train_dataset)
print(validation_dataset)
print(test_dataset)

## Dataset Reducing for limited Gpu

In [ ]:
SEED = 3407
TRAIN_SIZE = 10_000

train_dataset = processed_dataset["train"].shuffle(
    seed=SEED
).select(range(min(TRAIN_SIZE, len(processed_dataset["train"]))))

validation_dataset = processed_dataset["validation"]
test_dataset = processed_dataset["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Test examples:", len(test_dataset))

## Adding LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)

model.print_trainable_parameters()

In [ ]:
# psutil = Process & System Utilities
!pip install psutil

import time, psutil

# Clears unused GPU memory
# Frees cached tensors
# Prevents fake VRAM spikes
# Reset GPU stats

# peak GPU memory used
# This line resets peak counter so:
# You measure only THIS training run
# Not previous runs / notebooks

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
# your current Python notebook process
# Used to measure CPU RAM consumption
process = psutil.Process()
train_start_time = time.time()
cpu_ram_before = process.memory_info().rss / 1024**3  # GB

In [ ]:
process

In [ ]:
train_start_time

In [ ]:
cpu_ram_before

In [ ]:
!nvidia-smi

## Train with evaluation and checkpoint saving

In [ ]:
MAX_SEQ_LENGTH = 256

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=3407,
    data_seed=3407,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

train_result = trainer.train()

print(train_result)

Unsloth: Tokenizing ["text"] (num_proc=2): 100%
 10000/10000 [00:19<00:00, 975.20 examples/s]
Unsloth: Tokenizing ["text"] (num_proc=2): 100%
 1498/1498 [00:15<00:00, 101.38 examples/s]
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 625
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
 [625/625 36:44, Epoch 1/1]
Step	Training Loss	Validation Loss
250	0.852448	0.859324
500	0.839322	0.806060
625	0.877759	0.802162
Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in banglish-to-bangla-qwen2.5/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in banglish-to-bangla-qwen2.5/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in banglish-to-bangla-qwen2.5/checkpoint-625/tokenizer_config.json.
TrainOutput(global_step=625, training_loss=0.9468357078552246, metrics={'train_runtime': 2224.1407, 'train_samples_per_second': 4.496, 'train_steps_per_second': 0.281, 'total_flos': 1.0746373304223744e+16, 'train_loss': 0.9468357078552246, 'epoch': 1.0})

In [ ]:
!nvidia-smi

## Evaluate on the untouched test set and save the model

In [ ]:
# Final evaluation on the held-out test split
test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test",
)

print("Test metrics:", test_metrics)

In [ ]:
# Save the trained LoRA adapter and tokenizer
FINAL_ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"

os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print("Adapter saved to:", FINAL_ADAPTER_DIR)

### Test the Model

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def banglish_to_bangla(text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_INSTRUCTION,
        },
        {
            "role": "user",
            "content": text,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()


examples = [
    "ami ajke university jabo",
    "tumi kemon acho",
    "amar bangla vasha khub valo lage",
]

for example in examples:
    print("Banglish:", example)
    print("Bangla:", banglish_to_bangla(example))
    print("-" * 50)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Banglish: ami ajke university jabo
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Bangla: আমি আজকে ইউনিভার্সিটি যাবো
--------------------------------------------------
Banglish: tumi kemon acho
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Bangla: তুমি কেমন আছো
--------------------------------------------------
Banglish: amar bangla vasha khub valo lage
Bangla: আমার বাংলা ভাষা খুব ভালো লাগে
--------------------------------------------------

In [ ]:
import shutil

shutil.make_archive(
    "final_adapter",
    "zip",
    root_dir="./banglish-to-bangla-qwen2.5",
    base_dir="final_adapter"
)

In [ ]:
from google.colab import files

files.download("final_adapter.zip")